# Tutorial 11 — Supervised Fine-Tuning & LoRA

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part IV — Alignment**  
**Follows:** Tutorial 10 (Distributed Training)  
**Precedes:** Tutorial 12 (Direct Preference Optimization)

---

## What This Tutorial Covers

Pretraining produces a model that completes text. SFT produces a model
that follows instructions. The [gap between them is not about capability —
a pretrained model already has the knowledge. It is about *behavior*:]{.mark} the
pretrained model will continue a prompt in any plausible direction; the
SFT model has learned that a human question deserves a human answer.

This tutorial covers:

1. **The SFT objective** — what changes from pretraining, what stays the
   same. Why we mask the prompt tokens in the loss.
2. **Chat templates** — how conversations are formatted into token sequences.
   The system prompt, user turn, assistant turn. Special tokens.
3. **The SFT dataset** — `InstructDataset`: loading, formatting, packing
   vs padding, the [loss mask]{.underline}.
4. **Full fine-tuning** — running the pretraining loop on instruction data.
   Why it is prone to catastrophic forgetting.
5. **LoRA** — Low-Rank Adaptation. The math: why a weight update
   $\Delta W = BA$ where $B \in \mathbb{R}^{d \times r}$,
   $A \in \mathbb{R}^{r \times k}$ with $r \ll \min(d, k)$ captures
   most of the useful fine-tuning signal. How to inject LoRA into any
   `nn.Linear` layer. The `rank`, `alpha`, and `dropout` hyperparameters.
6. **Merging LoRA weights** — how to fold $\Delta W$ back into $W$ after
   training for zero-overhead inference.
7. **What to apply LoRA to** — the empirical answer from the original paper
   and subsequent work.

---

## 1. The SFT Objective

Pretraining minimizes cross-entropy over every token in the training corpus:

$$\mathcal{L}_{\text{pretrain}} = -\frac{1}{T} \sum_{t=1}^{T} \log p(x_t \mid x_{<t})$$

SFT minimizes cross-entropy over only the **response tokens** — the tokens
the assistant should generate. Prompt tokens are masked out of the loss:

$$\mathcal{L}_{\text{SFT}} = -\frac{1}{|\mathcal{R}|} \sum_{t \in \mathcal{R}} \log p(x_t \mid x_{<t})$$

where $\mathcal{R}$ is the set of positions corresponding to the assistant's
response.

**Why mask the prompt?** The model sees the prompt during the forward pass —
it needs to condition on it. [But we do not want to penalize it for not
"predicting" the user's question, which is given, not generated.]{.mark} Training
the model to predict prompt tokens pushes it toward completing prompts
rather than responding to them. Masking the prompt focuses the gradient
signal entirely on the quality of the response.

In PyTorch's `F.cross_entropy`, masking is implemented via `ignore_index`:

In [ ]:
loss = F.cross_entropy(
    logits.view(-1, vocab_size),
    targets.view(-1),
    ignore_index=-100,   # standard mask value
)
# Positions where targets == -100 contribute zero loss and zero gradient

---

## 2. Chat Templates

A chat template is a function that converts a list of conversation turns
into a single token sequence. The exact format varies by model family;
what matters is that it is consistent between training and inference.

We use a simple format for the nano model:

```
<|system|>
You are a helpful assistant.
<|user|>
What is the capital of France?
<|assistant|>
The capital of France is Paris.
<|end|>
```

Each special token (`<|system|>`, `<|user|>`, `<|assistant|>`, `<|end|>`)
needs to be in the tokenizer's vocabulary. For the nano tokenizer trained
on TinyShakespeare, we add them as special tokens.

In [ ]:
from dataclasses import dataclass
from typing import Literal

Role = Literal['system', 'user', 'assistant']

@dataclass
class Message:
    role:    Role
    content: str

SPECIAL_TOKENS = {
    'system':    '<|system|>',
    'user':      '<|user|>',
    'assistant': '<|assistant|>',
    'end':       '<|end|>',
}

def format_chat(
    messages:       list[Message],
    add_gen_prompt: bool = False,
) -> str:
    """
    Convert a list of messages to a single string using the chat template.

    add_gen_prompt=True appends '<|assistant|>\n' at the end — used during
    inference to prime the model to generate the assistant's response.
    """
    parts = []
    for msg in messages:
        role_token = SPECIAL_TOKENS[msg.role]
        parts.append(f"{role_token}\n{msg.content.strip()}\n{SPECIAL_TOKENS['end']}\n")

    text = ''.join(parts)
    if add_gen_prompt:
        text += f"{SPECIAL_TOKENS['assistant']}\n"
    return text


def build_loss_mask(
    tokens:    list[int],
    tokenizer,
) -> list[int]:
    """
    Build the loss mask for a formatted chat sequence.
    Returns a list of the same length as `tokens`:
    -  token_id  at assistant response positions (loss computed here)
    - -100       at all other positions (prompt, system, user turns — masked)
    """
    mask   = [-100] * len(tokens)
    text   = tokenizer.decode(tokens)

    # Find all assistant response spans
    asst_start_tok = tokenizer.encode(SPECIAL_TOKENS['assistant'])[0]
    end_tok        = tokenizer.encode(SPECIAL_TOKENS['end'])[0]

    in_response = False
    for i, tok in enumerate(tokens):
        if tok == asst_start_tok:
            in_response = True
            # mask the <|assistant|> token itself — model should not
            # "predict" the role marker, only the content after it
            continue
        if tok == end_tok and in_response:
            # Include the <|end|> token in the loss — model must learn to stop
            mask[i] = tok
            in_response = False
            continue
        if in_response:
            mask[i] = tok

    return mask

---

## 3. The SFT Dataset

In [ ]:
import json
import torch
from torch.utils.data import Dataset
from pathlib import Path

class InstructDataset(Dataset):
    """
    Loads instruction-following data from a JSONL file where each line is:
    {
        "messages": [
            {"role": "system",    "content": "You are a helpful assistant."},
            {"role": "user",      "content": "What is 2 + 2?"},
            {"role": "assistant", "content": "2 + 2 equals 4."}
        ]
    }

    Returns (input_ids, labels) pairs where:
    - input_ids: token IDs of the full conversation
    - labels:    token IDs at assistant positions, -100 elsewhere
    """

    def __init__(
        self,
        data_path:  str,
        tokenizer,
        max_length: int = 512,
        pad_to_max: bool = False,
    ):
        self.tokenizer   = tokenizer
        self.max_length  = max_length
        self.pad_to_max  = pad_to_max
        self.samples     = []

        with open(data_path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    self.samples.append(json.loads(line))
                except json.JSONDecodeError:
                    continue

        print(f"InstructDataset: {len(self.samples)} samples from {data_path}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        sample   = self.samples[idx]
        messages = [Message(**m) for m in sample['messages']]

        # Format to string
        text   = format_chat(messages)
        tokens = self.tokenizer.encode(text)

        # Truncate to max_length + 1 (need +1 for the shifted target)
        tokens = tokens[:self.max_length + 1]

        # Build loss mask
        labels = build_loss_mask(tokens, self.tokenizer)

        # Input is all tokens except the last; target is all except the first
        input_ids = tokens[:-1]
        labels    = labels[1:]   # shift: label[t] = token[t+1]

        # Pad if needed (for batching with DataLoader)
        length = len(input_ids)
        if self.pad_to_max:
            pad_len    = self.max_length - length
            input_ids  = input_ids  + [self.tokenizer.pad_id] * pad_len
            labels     = labels     + [-100]                  * pad_len

        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(labels,    dtype=torch.long),
        )


def collate_sft(batch):
    """
    Collate function for variable-length SFT samples.
    Pads to the longest sequence in the batch — avoids wasting compute
    padding all sequences to max_length when most are shorter.
    """
    input_ids, labels = zip(*batch)
    max_len = max(x.size(0) for x in input_ids)

    padded_inputs = torch.full((len(batch), max_len), 0,    dtype=torch.long)
    padded_labels = torch.full((len(batch), max_len), -100, dtype=torch.long)

    for i, (x, y) in enumerate(zip(input_ids, labels)):
        padded_inputs[i, :x.size(0)] = x
        padded_labels[i, :y.size(0)] = y

    return padded_inputs, padded_labels

### Generating a toy SFT dataset from TinyShakespeare

For the nano model, we construct a simple instruction dataset by treating
Shakespeare passages as "creative writing" responses:

In [ ]:
def make_shakespeare_instruct(
    raw_text: str,
    output_path: str,
    n_samples: int = 500,
):
    """
    Build a toy SFT dataset where:
    - user asks to "continue the passage" or "write in the style of Shakespeare"
    - assistant responds with a Shakespeare excerpt
    """
    import random
    random.seed(42)

    prompts = [
        "Continue this passage in the style of Shakespeare:",
        "Write a short dramatic monologue.",
        "Write dialogue between two characters.",
        "Complete this Shakespearean verse:",
        "Write a soliloquy.",
    ]

    # Split into ~200-token chunks
    words  = raw_text.split()
    chunks = [' '.join(words[i:i+80]) for i in range(0, len(words)-80, 80)]

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w') as f:
        for i in range(min(n_samples, len(chunks))):
            sample = {
                "messages": [
                    {"role": "system",    "content": "You are a creative writing assistant."},
                    {"role": "user",      "content": random.choice(prompts)},
                    {"role": "assistant", "content": chunks[i]},
                ]
            }
            f.write(json.dumps(sample) + '\n')

    print(f"Wrote {min(n_samples, len(chunks))} samples to {output_path}")

---

## 4. Full Fine-Tuning

Full fine-tuning trains all model parameters on the SFT dataset.
It is simple and effective when:

- Your SFT dataset is large (>100K samples)
- You have enough GPU memory to store the model + optimizer state
- Your SFT data distribution is not too far from pretraining

The training loop is nearly identical to pretraining, with two differences:
the dataset (`InstructDataset` instead of `PretrainingDataset`) and the
loss (masked cross-entropy instead of full cross-entropy).

In [ ]:
def sft_loss(logits, labels):
    """
    Masked cross-entropy: only compute loss where labels != -100.
    logits: (B, T, vocab_size)
    labels: (B, T) with -100 at masked positions
    """
    B, T, V = logits.shape
    return F.cross_entropy(
        logits.view(B * T, V),
        labels.view(B * T),
        ignore_index=-100,
    )

****Catastrophic forgetting.**** Full fine-tuning on a small SFT dataset can
overwrite the pretrained weights to the point where the model loses general
capabilities — it becomes good at responding to the specific prompts in your
SFT data but loses the breadth learned during pretraining.

The symptoms: eval loss on a *held-out pretraining corpus* increases during
SFT, even while SFT eval loss decreases. You are gaining instruction-following
ability but losing language modeling ability.

Mitigations:
- [Use a lower LR during SFT (typically 10× lower than pretraining LR)]{.underline}
- Use fewer epochs (1–3 over the SFT data, not 10+)
- Use LoRA — which introduces new parameters and barely moves the pretrained
  weights

---

## 5. LoRA: Low-Rank Adaptation

### The core idea

A pretrained weight matrix $W_0 \in \mathbb{R}^{d \times k}$ is frozen.
Fine-tuning is modeled as a low-rank update:

$$W = W_0 + \Delta W = W_0 + BA$$

where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$
with rank $r \ll \min(d, k)$.

During the forward pass, the output becomes:

$$h = W_0 x + \Delta W x = W_0 x + BAx$$

$W_0$ is frozen — it never receives gradients. Only $B$ and $A$ are trained.
The number of trainable parameters is:

$$|\theta_{\text{LoRA}}| = r(d + k) \quad \text{vs} \quad |\theta_{\text{full}}| = dk$$

For a typical Transformer hidden dimension $d = k = 768$ and $r = 8$:

$$\frac{r(d + k)}{dk} = \frac{8 \times 1536}{768^2} = \frac{12288}{589824} \approx 2\%$$

[LoRA uses approximately 2% of the parameters of full fine-tuning while
achieving comparable task performance on most benchmarks.]{.mark}

### Initialization

$A$ is initialized with a Gaussian (standard init for a linear layer).
[$B$ is initialized to **zero**. This means $\Delta W = BA = 0$ at the
start of training]{.underline} — the LoRA model begins as an exact copy of the
pretrained model. No warm-up is needed to stabilize the output; training
starts from a well-defined, well-understood point.

### The scaling factor $\alpha$

The actual update applied is:

$$\Delta W = \frac{\alpha}{r} BA$$

The $\alpha/r$ scaling factor controls the magnitude of the LoRA contribution
relative to the original weights. Setting $\alpha = r$ gives a scaling of 1.0
(LoRA update has the same scale as a full weight update). Setting $\alpha = 2r$
doubles the LoRA contribution. In practice, $\alpha = r$ or $\alpha = 2r$ are
the common choices — just keep $\alpha/r$ in $[1, 2]$.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class LoRALinear(nn.Module):
    """
    A drop-in replacement for nn.Linear with LoRA adaptation.

    The forward pass computes:
        h = W_0 x + (alpha/r) * B * A * x

    W_0 is frozen. Only A and B are trained.

    Args:
        in_features:  input dimension (k)
        out_features: output dimension (d)
        rank:         LoRA rank r (typical: 4, 8, 16, 32)
        alpha:        scaling factor (typical: rank or 2*rank)
        dropout:      dropout on the LoRA path (typical: 0.0 or 0.05)
        bias:         whether the original linear had a bias
    """

    def __init__(
        self,
        in_features:  int,
        out_features: int,
        rank:         int   = 8,
        alpha:        float = None,
        dropout:      float = 0.0,
        bias:         bool  = True,
    ):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.rank         = rank
        self.alpha        = alpha if alpha is not None else float(rank)
        self.scaling      = self.alpha / self.rank

        # Frozen pretrained weight
        self.weight = nn.Parameter(
            torch.empty(out_features, in_features), requires_grad=False
        )
        self.bias_param = nn.Parameter(
            torch.zeros(out_features), requires_grad=False
        ) if bias else None

        # Trainable LoRA matrices
        self.lora_A = nn.Parameter(torch.empty(rank, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))

        # Optional dropout on LoRA path
        self.lora_dropout = nn.Dropout(p=dropout) if dropout > 0 else nn.Identity()

        # Initialize A with kaiming uniform (same as nn.Linear default)
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        # B is already zero — ensures delta_W = 0 at init

    @classmethod
    def from_linear(
        cls,
        linear: nn.Linear,
        rank:   int   = 8,
        alpha:  float = None,
        dropout: float = 0.0,
    ) -> 'LoRALinear':
        """
        Create a LoRALinear from an existing nn.Linear, copying its weights.
        The original weights are frozen; new LoRA matrices are added.
        """
        has_bias = linear.bias is not None
        lora     = cls(
            linear.in_features, linear.out_features,
            rank=rank, alpha=alpha, dropout=dropout, bias=has_bias
        )
        lora.weight.data.copy_(linear.weight.data)
        if has_bias:
            lora.bias_param.data.copy_(linear.bias.data)
        return lora

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pretrained path (frozen)
        result = F.linear(x, self.weight, self.bias_param)
        # LoRA path (trainable)
        lora_out = F.linear(
            self.lora_dropout(x),
            self.lora_B @ self.lora_A,   # (out, rank) @ (rank, in) = (out, in)
        ) * self.scaling
        return result + lora_out

    def merge(self) -> nn.Linear:
        """
        Fold the LoRA update into the base weight and return a plain nn.Linear.
        Used before inference to eliminate the LoRA overhead.
        delta_W = scaling * B @ A
        W_merged = W_0 + delta_W
        """
        merged = nn.Linear(
            self.in_features, self.out_features,
            bias=self.bias_param is not None
        )
        merged.weight.data = self.weight.data + self.scaling * (self.lora_B @ self.lora_A)
        if self.bias_param is not None:
            merged.bias.data = self.bias_param.data
        return merged

    def extra_repr(self) -> str:
        return (f'in={self.in_features}, out={self.out_features}, '
                f'rank={self.rank}, alpha={self.alpha:.1f}, '
                f'scaling={self.scaling:.3f}')

---

## 6. Injecting LoRA into the Model

The standard approach is to replace target `nn.Linear` layers in the model
with `LoRALinear` layers. The original paper applied LoRA to the query and
value projection matrices in attention. Subsequent work shows that applying
it to all linear layers typically performs better for instruction following.

In [ ]:
def inject_lora(
    model:        nn.Module,
    rank:         int   = 8,
    alpha:        float = None,
    dropout:      float = 0.0,
    target_names: list[str] = None,
) -> nn.Module:
    """
    Replace nn.Linear layers with LoRALinear layers in-place.

    target_names: list of substrings. A layer is replaced if its name
    contains any of the substrings. If None, all Linear layers are replaced.

    Common choices:
    - ['q_proj', 'v_proj']             — original LoRA paper (attention only)
    - ['q_proj', 'k_proj', 'v_proj', 'o_proj']  — all attention projections
    - None (all linear layers)         — best for instruction following

    Returns the model with LoRA layers injected.
    """
    if target_names is None:
        target_names = ['']   # matches everything

    replaced = 0
    for name, module in list(model.named_modules()):
        # Find the parent module and attribute name for replacement
        parts  = name.split('.')
        parent = model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        attr = parts[-1]

        if (isinstance(module, nn.Linear) and
                any(t in name for t in target_names)):
            lora_layer = LoRALinear.from_linear(module, rank=rank,
                                                alpha=alpha, dropout=dropout)
            setattr(parent, attr, lora_layer)
            replaced += 1

    print(f"LoRA: replaced {replaced} Linear layers  (rank={rank}, alpha={alpha or rank})")
    return model


def freeze_base_model(model: nn.Module):
    """Freeze all parameters that are not LoRA matrices."""
    frozen = trained = 0
    for name, param in model.named_parameters():
        if 'lora_A' in name or 'lora_B' in name:
            param.requires_grad_(True)
            trained += param.numel()
        else:
            param.requires_grad_(False)
            frozen += param.numel()
    total = frozen + trained
    print(f"LoRA trainable: {trained:,} params  ({100*trained/total:.2f}% of {total/1e6:.1f}M)")


def count_trainable(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### Verifying the initialization

After injection, verify that:
1. LoRA matrices are the only trainable parameters
2. $\Delta W = 0$ at initialization (so output matches the pretrained model)

In [ ]:
# Load pretrained model
model = GPT(config)
model.load_state_dict(torch.load('runs/nano_gpt/best_checkpoint.pt')['model'])

# Test output before LoRA injection
x     = torch.randint(0, config.vocab_size, (2, 32))
with torch.no_grad():
    out_before = model(x)[0].clone()

# Inject LoRA
inject_lora(model, rank=8)
freeze_base_model(model)

# Test output after injection — should be identical since B=0
with torch.no_grad():
    out_after = model(x)[0]

max_diff = (out_before - out_after).abs().max().item()
print(f"Max output difference after LoRA injection: {max_diff:.2e}")
# Should be ~0 (floating point rounding only)
assert max_diff < 1e-5, "LoRA injection changed model output — B init is wrong"
print("✓ LoRA injection correct: output unchanged at initialization")

---

## 7. The LoRA Fine-Tuning Loop

The loop is nearly identical to pretraining, with three differences:
a smaller LR, fewer steps, and only LoRA parameters are updated.

In [ ]:
from torch.utils.data import DataLoader

def sft_train(
    model_path:   str,
    data_path:    str,
    output_dir:   str,
    rank:         int   = 8,
    alpha:        float = None,
    lora_dropout: float = 0.05,
    max_lr:       float = 2e-4,    # 10× lower than pretraining
    min_lr:       float = 2e-5,
    warmup_steps: int   = 50,
    max_steps:    int   = 1000,
    batch_size:   int   = 4,
    max_length:   int   = 256,
    eval_every:   int   = 100,
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dtype  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # ---- Load pretrained model ----
    from tutorial_02 import GPT, NanoGPTConfig
    from tutorial_03 import Tokenizer

    config = NanoGPTConfig()
    model  = GPT(config).to(device)
    ckpt   = torch.load(model_path, map_location=device)
    model.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
    print(f"Loaded pretrained model from {model_path}")

    # ---- Inject LoRA ----
    inject_lora(model, rank=rank, alpha=alpha, dropout=lora_dropout)
    freeze_base_model(model)

    # ---- Data ----
    tok = Tokenizer.load('nano_tokenizer.json')

    train_ds = InstructDataset(data_path, tok, max_length=max_length)
    val_size  = max(1, len(train_ds) // 10)
    train_ds, val_ds = torch.utils.data.random_split(
        train_ds, [len(train_ds) - val_size, val_size]
    )

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        collate_fn=collate_sft, num_workers=2, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        collate_fn=collate_sft, num_workers=1
    )

    # ---- Optimizer — only LoRA parameters ----
    lora_params = [p for p in model.parameters() if p.requires_grad]
    optimizer   = torch.optim.AdamW(lora_params, lr=max_lr, weight_decay=0.01)
    scheduler   = make_cosine_schedule(optimizer, max_lr, min_lr,
                                        warmup_steps, max_steps)

    # ---- Training loop ----
    model.train()
    train_iter = iter(train_loader)
    best_eval  = float('inf')

    for step in range(max_steps):
        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            x, y = next(train_iter)

        x, y = x.to(device), y.to(device)

        with torch.autocast(device_type=device.type, dtype=dtype):
            logits, _ = model(x)   # get logits; ignore built-in loss
            loss      = sft_loss(logits, y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lora_params, 1.0)
        optimizer.step()
        scheduler.step()

        if step % 50 == 0:
            lr = optimizer.param_groups[0]['lr']
            print(f"step {step:4d}  loss={loss.item():.4f}  lr={lr:.2e}")

        # ---- Eval ----
        if step % eval_every == 0 and step > 0:
            model.eval()
            eval_losses = []
            with torch.no_grad():
                for xv, yv in val_loader:
                    with torch.autocast(device_type=device.type, dtype=dtype):
                        logits_v, _ = model(xv.to(device))
                        eval_losses.append(
                            sft_loss(logits_v, yv.to(device)).item()
                        )
            eval_loss = float(np.mean(eval_losses))
            print(f"  → eval_loss={eval_loss:.4f}")
            model.train()

            if eval_loss < best_eval:
                best_eval = eval_loss
                # Save only LoRA weights — much smaller than the full model
                lora_state = {
                    name: param
                    for name, param in model.state_dict().items()
                    if 'lora_A' in name or 'lora_B' in name
                }
                torch.save({
                    'lora_state': lora_state,
                    'rank': rank, 'alpha': alpha or rank,
                    'step': step, 'eval_loss': eval_loss,
                }, f'{output_dir}/lora_best.pt')

    print(f"\nSFT complete. Best eval loss: {best_eval:.4f}")
    print(f"LoRA weights saved to {output_dir}/lora_best.pt")

---

## 8. Merging LoRA Weights

Before serving the model, merge $\Delta W$ back into $W_0$:

$$W_{\text{merged}} = W_0 + \frac{\alpha}{r} BA$$

[After merging, the model is a plain GPT with no LoRA overhead. Inference
is identical to the pretrained model in speed and memory.]{.mark}

In [ ]:
def merge_lora(model: nn.Module) -> nn.Module:
    """
    Replace all LoRALinear layers with merged nn.Linear layers.
    Returns the model with LoRA folded into the base weights.
    """
    merged_count = 0
    for name, module in list(model.named_modules()):
        if not isinstance(module, LoRALinear):
            continue
        parts  = name.split('.')
        parent = model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        attr   = parts[-1]
        setattr(parent, attr, module.merge())
        merged_count += 1

    print(f"Merged {merged_count} LoRA layers into base weights")
    return model


def load_lora_checkpoint(
    model:      nn.Module,
    lora_path:  str,
    rank:       int   = 8,
    alpha:      float = None,
):
    """
    Load a saved LoRA checkpoint into a model.
    Injects LoRA, loads the saved lora_A/lora_B weights.
    """
    ckpt = torch.load(lora_path, map_location='cpu')
    inject_lora(model, rank=ckpt.get('rank', rank),
                alpha=ckpt.get('alpha', alpha))
    # Load only the LoRA parameters
    missing, unexpected = model.load_state_dict(
        ckpt['lora_state'], strict=False
    )
    print(f"LoRA loaded: {len(ckpt['lora_state'])} tensors")
    if unexpected:
        print(f"  Unexpected keys: {unexpected}")
    return model

---

## 9. What to Apply LoRA To

The original LoRA paper (Hu et al., 2021) applied LoRA to the query and
value projections in attention only. Subsequent empirical work found:

| Target layers | Relative performance | Trainable params |
|---|---|---|
| Q, V only | Baseline | ~1% |
| Q, K, V, O | +2–5% on instruction tasks | ~2% |
| All attention + FFN | Best for instruction following | ~4% |
| All linear layers | Marginal improvement over above | ~6% |

For the nano model, the distinction matters less (the model is small enough
that full fine-tuning is feasible). For large models (7B+), the parameter
efficiency of LoRA is the enabling factor.

**Rank selection:** $r = 8$ is a good default.[^lora_rank]

[^lora_rank]: Use $r=4$ for narrow task adaptation (e.g., style transfer on a small dataset). Use $r=16$–$32$ when the fine-tuning distribution diverges significantly from pretraining (e.g., a new domain or language). There is rarely a reason to exceed $r=64$ — at that scale, full fine-tuning with a lower LR is often simpler and equally effective. Lower ranks ($r = 4$) work
for narrow task adaptation. Higher ranks ($r = 16, 32$) help when the
fine-tuning distribution is very different from pretraining. There is
rarely a reason to go above $r = 64$.

---

## Summary

| Concept | Key detail |
|---|---|
| SFT loss masking | `ignore_index=-100` at prompt positions. Gradient only from response tokens. |
| Chat template | Consistent format between training and inference. Special role tokens in vocab. |
| Label shift | `targets[t]` = `tokens[t+1]`. Build mask on full sequence, then shift both. |
| Full fine-tuning risk | Catastrophic forgetting on small datasets. Use low LR (10×) and few epochs. |
| LoRA rank-decomposition | $\Delta W = BA$, $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$, $r \ll \min(d,k)$. |
| B initialized to zero | $\Delta W = 0$ at init. Output identical to pretrained model at step 0. |
| $\alpha/r$ scaling | Controls LoRA update magnitude. Keep in $[1, 2]$. Default: $\alpha = r$. |
| Trainable param fraction | $r(d+k)/(dk) \approx 2\%$ for $r=8$, $d=k=768$. |
| Save only LoRA weights | `lora_A` + `lora_B` keys only — a fraction of full model size. |
| Merge before inference | $W_{\text{merged}} = W_0 + (\alpha/r)BA$. Zero inference overhead after merge. |
| Best target layers | All attention + FFN linears for instruction following. Q+V only for narrow tasks. |

---

## Exercises

**1.** Verify the loss mask implementation: create a short multi-turn
conversation, tokenize it, build the loss mask, and print the decoded
text for each position with its mask value. Confirm that all user/system
tokens have `mask=-100` and all assistant tokens have `mask=token_id`.

**2.** Run the LoRA training loop for 500 steps with `rank` ∈ {4, 8, 16}.
Plot eval loss vs step for each rank. Verify that higher rank reaches
lower eval loss but requires more steps to stabilize (due to more
parameters to optimize from zero).

**3.** Implement the **forgetting check**: before SFT, evaluate the pretrained
model's perplexity on a held-out chunk of TinyShakespeare. After SFT
(full fine-tuning, not LoRA), evaluate again. Report the perplexity
increase. Then repeat with LoRA — confirm that LoRA causes significantly
less forgetting.

**4.** Implement `LoRALinear.merge()` and verify correctness: inject LoRA,
train for 100 steps, then merge. The merged model's output on a test
batch must match the unmerged LoRA model's output to within floating
point precision (`< 1e-5` max absolute difference).

**5.** Extend `inject_lora` to support **per-layer rank**: instead of a single
`rank` argument, accept a `rank_map: dict[str, int]` where keys are
layer name substrings and values are the rank to use for matching layers.
Use this to assign `rank=16` to the first and last Transformer blocks
(which tend to be most important for task adaptation) and `rank=4`
to middle blocks.

**6.** Add a `lora_stats()` function that, given a model with LoRA layers,
reports: total trainable params, total frozen params, trainable fraction,
per-layer rank, and the current $\|\Delta W\|_F / \|W_0\|_F$ ratio for
each LoRA layer. This ratio measures how much the LoRA adaptation has
moved each layer from its pretrained initialization — a high ratio late
in training may indicate overfitting to the SFT data.